In [ ]:
import json
with open('../Data_Files/wanted_detail_improve_20250604.json','r', encoding='utf-8') as f:
    job_postings = json.load(f)

for job_postings in job_postings.values():
    title = job_postings.get("제목","무제")
    intro = job_postings.get("회사 소개",''),
    main_task = job_postings.get("주요 업무", []),
    requirements = job_postings.get("자격 요건", []),
    preferred_points = job_postings.get("우대 사항", []),
    benefits = job_postings.get("복지", ''),
    hire_rounds = job_postings.get("채용절차", ''),
    attraction = job_postings.get("장점", ''),
    company = job_postings.get("회사명", ''),
    location = job_postings.get("근무지", ''),
    district = job_postings.get("지역", ''),
    category_parent = job_postings.get("직군",''),
    category_child = job_postings.get("직무", ''),
    skill = job_postings.get("기술 스택", []),
    annual_from = job_postings.get("요구 최소 경력", ""),
    annual_to = job_postings.get("요구 최대 경력", ""),
    is_newbie = job_postings.get("신입 지원 가능 여부", ""),
    due_time = job_postings.get("마감일", "상시 공고"),

    print(title)


In [ ]:
import pandas as pd
df = pd.read_json('../Data_Files/wanted_detail_improve_20250604.json', orient='index')


df


In [ ]:
job_postings = df.to_dict(orient='records')

job_postings[0].get("제목","무제")

In [66]:
import json
import os
from Run_Pipeline.Env_Loader import EnvLoader
import openai

# 1) OpenAI API 키 불러오기 (환경 변수)
EnvLoader.load_local('../.env')

'''
채용공고 json 파일을 불러와서 , 해당 파일을 기준으로 프롬프트 설정을해서 QA 세트를 생성한다
그리고, 생성된 QA 세트를 Gemma3 학습 포맷에 맞춰 input/target 형태로 변환하여 JSONL 파일로 저장한다.

채용공고 JSON 파일은 다음과 같은 구조를 가집니다:
{
    "제목": "채용공고 제목",
    "회사 소개": "회사에 대한 소개",
    "주요 업무": ["업무1", "업무2", ...],
    "우대 사항": ["우대사항1", "우대사항2", ...]
}

QA세트는 너무 일반적이지 않고, 채용공고의 특성을 반영하여 생성됩니다.

'''

client = openai.OpenAI()


def generate_qa_pairs_for_posting(job_postings: dict) -> list:
    """
    GPT-4o-mini로 QA 쌍을 생성하는 부분은 이전 예시와 동일합니다.
    반환값: [{"question": "...", "answer": "..."}, ...]
    """
    title = job_postings.get("제목","무제")
    intro = job_postings.get("회사 소개",''),
    main_task = job_postings.get("주요 업무", []),
    requirements = job_postings.get("자격 요건", []),
    preferred_points = job_postings.get("우대 사항", []),
    benefits = job_postings.get("복지", ''),
    hire_rounds = job_postings.get("채용절차", ''),
    attraction = job_postings.get("장점", ''),
    company = job_postings.get("회사명", ''),
    location = job_postings.get("근무지", ''),
    district = job_postings.get("지역", ''),
    category_parent = job_postings.get("직군",''),
    category_child = job_postings.get("직무", ''),
    skill = job_postings.get("기술 스택", []),
    annual_from = job_postings.get("요구 최소 경력", ""),
    annual_to = job_postings.get("요구 최대 경력", ""),
    is_newbie = job_postings.get("신입 지원 가능 여부", ""),
    due_time = job_postings.get("마감일", "상시 공고"),

    prompt = f"""
아래는 하나의 채용공고 정보입니다.
'직무', '주요 업무', '우대사항', '회사 소개','기술스택' 부분만 이용해서
질문(Question)과 답변(Answer) 쌍을 JSON 배열 형식으로 생성해 주세요.
질문은 실제 신입/주니어 레벨의 사용자가 주로 질문하는 형태로 질문을 작성합니다.
답변(A)은 채용공고 정보를 참고하되, 연봉·마감일 등 가변적 수치는 생략하고, 해당 기업의 실무자나 인사담당자가 전문적으로 답변하는 형태로 작성합니다.
출력 예시:
[
  {{
    "question": "질문 텍스트",
    "answer": "답변 텍스트"
  }},
  ...
]

---
채용공고 정보:
제목: {title}

회사 소개:
{intro}

주요 업무:
{main_task}

우대사항:
{preferred_points}

자격 요건:
{requirements}

복지:
{benefits}

기술 스택:
{skill}

신입 지원 가능 여부:
{is_newbie}

요구 최소 경력:
{annual_from}

---

위 내용을 바탕으로 4~6개의 일관된 QA 쌍을 만들어 주세요.
JSON 배열 형태로, 공백이나 주석 없이 순수한 JSON만 반환해 주세요.
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "당신은 채용 공고를 바탕으로 QA 쌍을 생성하는 전문가입니다."},
            {"role": "user",   "content": prompt}
        ],
        temperature=0.0,
        max_tokens=800,
        top_p=1.0,
    )
    generated = response.choices[0].message.content.strip()

    try:
        qa_list = json.loads(generated)
    except json.JSONDecodeError:
        start = generated.find("[")
        end   = generated.rfind("]") + 1
        qa_list = json.loads(generated[start:end])

    return qa_list

In [67]:
qa_list = generate_qa_pairs_for_posting(job_postings[0])

In [68]:
qa_list

[{'question': '프로모션 기획 마케터의 주요 업무는 무엇인가요?',
  'answer': '프로모션 기획 마케터는 사업 목표 달성을 위한 온사이트 프로모션 전략을 수립하고 실행하며, 월 단위의 주요 프로모션 테마를 기획합니다. 또한, 프로모션 이벤트의 기획 및 운영, 성과 분석을 통해 개선안을 도출하는 역할을 맡고 있습니다.'},
 {'question': '이 직무에 지원하기 위해 필요한 경험은 무엇인가요?',
  'answer': '최소 1년 이상의 유관 경험이 필요하며, 경력의 절반 이상을 이커머스 산업에서 재직한 경험이 있어야 합니다. 또한, 이커머스 MD 또는 마케팅 경험이 1년 이상 있어야 하며, 프로모션을 통해 매출을 개선한 경험이 요구됩니다.'},
 {'question': '우대사항에는 어떤 것들이 있나요?',
  'answer': '우대사항으로는 Amplitude 활용 가능, 서비스 마케팅 및 사업 전략까지 확장 가능한 역량, 프로모션 기획서 작성 및 카피라이팅 능력, IT, 이커머스, 마케팅, 라이프스타일의 최신 트렌드에 대한 이해가 포함됩니다.'},
 {'question': '이 직무에서 요구하는 스킬은 무엇인가요?',
  'answer': '유관 부서와 원활하고 명확한 커뮤니케이션 역량, 데이터 기반으로 한 정량적인 근거 제시 능력, 그리고 꼼꼼한 업무처리 능력이 요구됩니다.'},
 {'question': '회사의 복지 혜택은 어떤 것이 있나요?',
  'answer': '컬리는 최적의 근무 환경을 제공하며, 매월 유급 반차와 장기근속에 대한 유급휴가 및 휴가비 지원 제도를 운영합니다. 또한, 의료비 지원, 모션 데스크 기본 지원, 프라이빗 심리 상담 지원 등 다양한 복지 혜택을 제공합니다.'}]

In [70]:
def wrap_for_gemma3(question: str, answer: str) -> dict:
    """
    Gemma3 학습 포맷에 맞춰 input/target 을 생성해 줍니다.
    """
    # 1) input 문자열 생성
    input_text = (
        "<bos>\n"
        "<start_of_turn>user\n"
        "시스템: 너는 채용 공고 정보를 기반으로 QA를 수행하는 챗봇입니다.\n"
        "<end_of_turn>\n"
        "<start_of_turn>user\n"
        f"질문: {question}\n"
        "<end_of_turn>\n"
        "<start_of_turn>model"
    )

    # 2) target 문자열 생성
    target_text = f"{answer}\n<end_of_turn><eos>"

    return {"input": input_text, "target": target_text}

In [71]:
wr = wrap_for_gemma3(qa_list[0].get("question", ""), qa_list[0].get("answer", ""))
wr

{'input': '<bos>\n<start_of_turn>user\n시스템: 너는 채용 공고 정보를 기반으로 QA를 수행하는 챗봇입니다.\n<end_of_turn>\n<start_of_turn>user\n질문: 프로모션 기획 마케터의 주요 업무는 무엇인가요?\n<end_of_turn>\n<start_of_turn>model',
 'target': '프로모션 기획 마케터는 사업 목표 달성을 위한 온사이트 프로모션 전략을 수립하고 실행하며, 월 단위의 주요 프로모션 테마를 기획합니다. 또한, 프로모션 이벤트의 기획 및 운영, 성과 분석을 통해 개선안을 도출하는 역할을 맡고 있습니다.\n<end_of_turn><eos>'}

In [ ]:
if __name__ == "__main__":
    # 3) 로컬 채용공고 JSON 불러오기 (리스트 형태)
    with open("job_postings.json", "r", encoding="utf-8") as f:
        job_postings = json.load(f)

    # 4) 전체 결과 저장용 리스트
    gemma3_data = []

    for posting in job_postings:
        title = posting.get("제목", "무제 공고")
        print(f"▶ QA 생성 중: {title}")
        qa_pairs = generate_qa_pairs_for_posting(posting)

        # 5) 각 QA 쌍마다 Gemma3 input/target 생성
        for qa in qa_pairs:
            q = qa.get("question", "").strip()
            a = qa.get("answer", "").strip()
            if not q or not a:
                continue
            wrapped = wrap_for_gemma3(q, a)
            gemma3_data.append(wrapped)

    # 6) JSONL 포맷으로 저장
    out_path = "job_postings_gemma3.jsonl"
    with open(out_path, "w", encoding="utf-8") as fout:
        for entry in gemma3_data:
            fout.write(json.dumps(entry, ensure_ascii=False) + "\n")

    print(f"\n✅ Gemma3 형식 JSONL 생성 완료 → {out_path}")